# Improved — retrieve-then-rerank with a local LLM

Baseline **+** a local Qwen3-8B that reranks the retrieved codes. The 8B model loads **4-bit** (~6 GB) so it fits a free Colab **T4**.

See `docs/03_improved.md`.

## 1. Setup

Clone the `course` branch and install. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [1]:
!git clone https://github.com/duongtruongbinh/viettel_ai_race_task2 medextract
%cd medextract
!pip -q install -r requirements.txt
!pip -q install -e .

fatal: destination path 'medextract' already exists and is not an empty directory.
/home/binhdt/viettel-ai/notebooks/medextract


## 2. Knowledge bases

The linking step needs the RxNorm + ICD-10 knowledge bases. Their source files are license-gated, so download them yourself (see `INSTALL.md`) and place them in `data/kb/raw/`, then build the indexes.

> If you already have prebuilt `data/kb/processed/*.parquet` + `*.faiss` (e.g. from Google Drive), copy them into `data/kb/processed/` and skip the build cell.

In [2]:
# after placing the raw sources in data/kb/raw/ :
!python -m medextract.kb.build_rxnorm
!python -m medextract.kb.build_icd
!python -m medextract.kb.index --device auto

rxnorm_terms: 107,061 rows, 107,061 rxcui -> data/kb/processed/rxnorm_terms.parquet
tty
SCD     40027
SCDC    28326
SBD     24060
IN      14648
  308135: ['amlodipine 10 MG Oral Tablet']
  243670: ['aspirin 81 MG Oral Tablet']
  866436: ['24 HR metoprolol succinate 50 MG Extended Release Oral Tablet']
  392085: ['guaifenesin 800 MG Oral Tablet']
  313782: ['acetaminophen 325 MG Oral Tablet']
  904475: ['pravastatin sodium 40 MG Oral Tablet']
  197527: ['clonazepam 0.5 MG Oral Tablet']
[icd] using Vietnamese source: data/kb/raw/Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx
icd_terms: 15,846 rows, 15,844 codes -> data/kb/processed/icd_terms.parquet
source
QD4469        15844
QD98-COVID        2
  K21.0: ['Bệnh trào ngược dạ dày - thực quản kèm viêm thực quản']
  K21.9: ['Bệnh trào ngược dạ dày - thực quản không kèm viêm thực quản']
  I10: ['Bệnh tăng huyết áp vô căn (nguyên phát)']
  E11.9: ['Bệnh đái tháo đường típ 2, không kèm biến chứng']
  U07.1: ['COVID-19, virus được xác định']


## 3. Run the improved pipeline on the sample notes

First run downloads Qwen3-8B (a few minutes). `configs/improved.yaml` sets `load_in_4bit: true`.

In [3]:
!python run.py --config configs/improved.yaml --input data/sample_input --output out/demo_imp

2026-07-18 15:21:52,155 INFO medextract.llm.engine: loading LLM /mnt/pretrained_fm/Qwen_Qwen3-8B on cuda:1 (bfloat16, 4-bit)
Loading weights: 100%|███████████████████████| 399/399 [00:35<00:00, 11.29it/s]
2026-07-18 15:22:31,220 INFO medextract.llm.engine: LLM parameter count: 4.72B
2026-07-18 15:22:31,289 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:2
2026-07-18 15:22:32,423 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:22:32,451 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-18 15:22:32,738 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.

## 4. Score baseline vs improved on the dev set

In [4]:
!python run.py --config configs/improved.yaml --input data/dev/input --output out/dev_imp
!python score.py --pred out/dev_imp --gold data/dev

2026-07-18 15:23:01,681 INFO medextract.llm.engine: loading LLM /mnt/pretrained_fm/Qwen_Qwen3-8B on cuda:1 (bfloat16, 4-bit)
Loading weights: 100%|██████████████████████| 399/399 [00:03<00:00, 105.76it/s]
2026-07-18 15:23:09,269 INFO medextract.llm.engine: LLM parameter count: 4.72B
2026-07-18 15:23:09,337 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:2
2026-07-18 15:23:09,691 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:23:09,720 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-18 15:23:10,003 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.

## 5. Build a submission zip

Point `--input` at the official test directory; `--zip` writes `out/sub/submission.zip`.

In [5]:
!python run.py --config configs/improved.yaml --input data/sample_input --output out/sub --zip
!ls -la out/sub/submission.zip

2026-07-18 15:24:06,574 INFO medextract.llm.engine: loading LLM /mnt/pretrained_fm/Qwen_Qwen3-8B on cuda:1 (bfloat16, 4-bit)
Loading weights: 100%|██████████████████████| 399/399 [00:03<00:00, 105.05it/s]
2026-07-18 15:24:14,099 INFO medextract.llm.engine: LLM parameter count: 4.72B
2026-07-18 15:24:14,166 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:2
2026-07-18 15:24:14,557 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:24:14,585 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-18 15:24:14,879 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.